In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE  Silver_FleetSensorEvents 
TBLPROPERTIES (
    'delta.feature.allowColumnDefaults' = 'supported'
)
AS
SELECT
    EventID,
    VehicleID,
    Timestamp,
    Latitude,
    Longitude,
    Speed,
    FuelLevel,
    EngineTemp,
    Odometer,
    GeoFenceID,
    GeoFenceStatus,
    DoorStatus,
    AuxEquipment1Status,
    AuxEquipment2Status,
    AuxEquipment3Status,
    AuxEquipment4Status,
    BatteryVoltage,
    TirePressureFrontLeft,
    TirePressureFrontRight,
    TirePressureRearLeft,
    TirePressureRearRight,
    Temperature,
    Humidity,
    current_timestamp() AS ingest_timestamp,
    'Bronze_FleetSensorEvents' AS source_system,
    
    -- Enrichment using UDFs
    IsOverSpeeding(Speed, 100) AS OverSpeedAlert,
    IsLowFuel(FuelLevel) AS FuelStatus,
    TirePressureStatus(TirePressureFrontLeft, TirePressureFrontRight,
                       TirePressureRearLeft, TirePressureRearRight) AS TireStatus,
    EngineTempStatus(EngineTemp) AS EngineStatus,
    VehicleRiskScore(
        Speed,
        FuelLevel,
        TirePressureFrontLeft < 28 OR TirePressureFrontLeft > 36
        OR TirePressureFrontRight < 28 OR TirePressureFrontRight > 36
        OR TirePressureRearLeft < 28 OR TirePressureRearLeft > 36
        OR TirePressureRearRight < 28 OR TirePressureRearRight > 36,
        EngineTemp,
        GeoFenceStatus
    ) AS RiskScore
FROM Bronze_FleetSensorEvents;
